# Facial Action Units — YOLO11 + OpenGraphAU

| Stage | Model | Output |
|---|---|---|
| Face detection | `YoloFace11Detector` | bounding box + face crop |
| AU detection | `OpenGraphAuWrapper` | 41-dim intensity vector |

`OpenGraphAuWrapper` accepts a raw `(B, 3, H, W)` uint8 tensor and returns a `(B, 41)` float tensor covering standard AU codes plus lateralised variants.
Demo uses `face.jpg` (single frontal face).

In [1]:
from pathlib import Path

from exordium.video.core.io import image_to_np
from exordium.video.face.detector.yolo11 import YoloFace11Detector

face_path = Path("../tests/fixtures/image/face.jpg")
face_rgb = image_to_np(face_path, channel_order="RGB")  # (H, W, 3) uint8

# Detect face and get a tight square crop.
# crop() returns a (3, H', W') uint8 RGB tensor.
detector = YoloFace11Detector(device_id=None, conf=0.5)
frame_dets = detector.detect_image(face_rgb)
face_crop = frame_dets.get_detection_with_highest_score().crop(square=True)  # (3, H', W') tensor

# OpenGraphAU expects (B, 3, H, W) uint8 tensor.
face_tensor = face_crop.unsqueeze(0)  # (1, 3, H', W')
print(f"Face tensor: {tuple(face_tensor.shape)}  dtype={face_tensor.dtype}")

/Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-23 11:44:18 INFO YoloFace11Detector loaded 'yolo11n-pose_widerface' on cpu (conf=0.5).


Face tensor: (1, 3, 700, 700)  dtype=torch.uint8


## 1. Detect Action Units with OpenGraphAU

In [2]:
from exordium.video.face.au import OpenGraphAuWrapper

# stage=2 (default) uses the 2-stage SwinT weights (higher accuracy)
# stage=1 uses the 1-stage SwinT weights
au_model = OpenGraphAuWrapper(stage=2, device_id=-1)  # -1 → CPU

# __call__ accepts tensor (B, 3, H, W) uint8 → Tensor (B, 41)
au_feats = au_model(face_tensor)  # Tensor (1, 41)

print(f"AU features: {tuple(au_feats.shape)}  dtype={au_feats.dtype}")
print(f"Value range : [{au_feats.min():.3f}, {au_feats.max():.3f}]")

AU features: (1, 41)  dtype=torch.float32
Value range : [-0.020, 0.552]


## 2. AU Registry

In [3]:
from exordium.video.face.au import AU_ids, AU_names

intensities = au_feats[0]  # (41,) Tensor

print(f"Total action units: {len(AU_ids)}")
print()
print(f"{'AU':5s}  {'Name':35s}  {'Intensity':>9s}")
print("-" * 56)
for au_id, au_name, v in zip(AU_ids, AU_names, intensities.tolist()):
    bar = "#" * int(v * 20)
    print(f"{au_id:5s}  {au_name:35s}  {v:.3f}  {bar}")

Total action units: 41

AU     Name                                 Intensity
--------------------------------------------------------
1      Inner brow raiser                    0.088  #
2      Outer brow raiser                    0.098  #
4      Brow lowerer                         0.146  ##
5      Upper lid raiser                     0.377  #######
6      Cheek raiser                         0.292  #####
7      Lid tightener                        0.097  #
9      Nose wrinkler                        0.093  #
10     Upper lip raiser                     0.078  #
11     Nasolabial deepener                  0.216  ####
12     Lip corner puller                    0.268  #####
13     Sharp lip puller                     0.304  ######
14     Dimpler                              0.241  ####
15     Lip corner depressor                 -0.020  
16     Lower lip depressor                  0.307  ######
17     Chin raiser                          0.201  ####
18     Lip pucker                   

## 3. Active AUs

In [10]:
threshold = 0.5
active = [
    (au_id, au_name, float(v))
    for au_id, au_name, v in zip(AU_ids, AU_names, intensities.tolist())
    if float(v) > threshold
]

if active:
    print(f"Active AUs (intensity > {threshold}):")
    for au_id, au_name, v in sorted(active, key=lambda x: x[2], reverse=True):
        print(f"  {au_id:5s}  {au_name:35s}  {v:.3f}")
else:
    print("No AUs above threshold.")

Active AUs (intensity > 0.5):
  25     Lips part                            0.552
  18     Lip pucker                           0.500


## 4. Batch Processing

In [5]:
# Stack multiple face tensors into a batch (B, 3, H, W)
import torchvision.transforms.functional as TF

# Resize all crops to the same spatial size before stacking
crop_resized = TF.resize(face_tensor, (224, 224))  # (1, 3, 224, 224)
batch_tensor = crop_resized.repeat(3, 1, 1, 1)  # (3, 3, 224, 224)

batch_feats = au_model(batch_tensor)  # (3, 41) Tensor

print(f"Batch AU features: {tuple(batch_feats.shape)}  (3 faces × {batch_feats.shape[1]} AUs)")

Batch AU features: (3, 41)  (3 faces × 41 AUs)
